# 04.3 Fractionation: Exercises

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Model imports, parameter changes, fraction cuts and plotting.

**Practises:** collection windows, purity, recovery and `FractionationOptimizer`.

1. Choose a collection window
2. Define protein purity
3. Optimize the collection window

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

In [ ]:
%matplotlib inline

Collect **protein C** from the supplied [load-wash-elute process](https://cadet-process.readthedocs.io/en/latest/examples/load_wash_elute/lwe_flow_rate.html). First choose a collection window, then let the optimizer improve recovery at **95% protein purity**.

```{tip}
Run the cell below once with **Shift+Enter** to load the prebuilt process and plotting tools for this exercise.
```

In [ ]:
from examples.load_wash_elute.lwe_flow_rate import process
from CADETProcess.simulator import Cadet
from CADETProcess.fractionation import Fractionator, FractionationOptimizer
from CADETProcess.plotting import SecondaryAxis

## Exercise 1: choose a collection window

The supplied cell simulates the process and plots salt on a secondary axis. Choose a start and end time for collecting **C**; try **5–8 min** first. Use `add_fractionation_event()` with target `"C"` to collect and `-1` to return to waste. Event times are in **seconds**.

In [ ]:
# Simulate once; the collection calculations reuse these results.
simulator = Cadet()
simulation_results = simulator.simulate(process)
sec = SecondaryAxis(
    components=["Salt"],
    ylabel="Salt concentration / mM",
)
_ = simulation_results.solution.outlet.outlet.plot(secondary_axes=sec)

In [ ]:
# Fractionator with the collection window for C


In [ ]:
# Show the collection window and the performance of the collected pools.
fig, ax = fractionator.plot_fraction_signal(components=["A", "B", "C"])
# Add salt on its own axis alongside the protein collection windows.
salt_axis = ax.twinx()
_ = simulation_results.solution.outlet.outlet.plot(
    components=["Salt"], ax=salt_axis, show_legend=False,
)
salt_axis.lines[0].set(color="gray", linestyle="--")
_ = salt_axis.legend(loc="upper left")
_ = salt_axis.set_ylabel("Salt concentration / mM")
fig.set_size_inches(8, 4)
print(fractionator.performance)

```{dropdown} Interpretation
Recovery shows how much of the injected C the window collects. Purity is very low here: by default it counts every component in the collected liquid, and salt is present at much higher concentrations than the proteins.
```

## Exercise 2: define protein purity

Salt is part of the collected liquid, but our product specification concerns **C relative to proteins A, B and C**. Create a new `Fractionator` with `components=["A", "B", "C"]`, keeping the same collection window. Compare C's purity and recovery with Exercise 1.

```{tip}
Use the name `"C"` again. It identifies the same protein even when excluding salt changes the component order. Filtering the fractionation also filters its plot; the salt axis is no longer needed.
```

In [ ]:
# Fractionator for the proteins only


In [ ]:
_ = protein_fractionator.plot_fraction_signal()
print(protein_fractionator.performance)

```{dropdown} Interpretation
With salt excluded, purity measures C relative to proteins A, B and C, which matches the product specification. Recovery does not change: the window and the collected amount of C are the same.
```

## Exercise 3: optimize the collection window

Use `optimize_fractionation()` with the same protein selection and `purity_required=[0, 0, 0.95]`. The entries refer to **A, B and C**: only C is collected as a product. Compare its recovery and collection window with your manual choice.

The supplied optimizer uses `catol=1e-4`, allowing a purity violation of at most **0.01 percentage points**. Inspect the achieved purity rather than rounding it to 95%. For a strict specification, request a small safety margin, for example **0.951**.

In [ ]:
fractionation_optimizer = FractionationOptimizer()
fractionation_optimizer.optimizer.catol = 1e-4

In [ ]:
# Optimize the collection window for C


In [ ]:
# Display the achieved performance and the optimized protein fractions.
print(optimized_fractionator.performance)
_ = optimized_fractionator.plot_fraction_signal()

```{dropdown} Interpretation
The optimizer maximizes the collected amount of C while keeping its purity at or above 95%. If your manual window was below 95% purity, the optimized window is narrower; if it was above, the optimizer can widen it to collect more C. Check the achieved purity against the specification rather than its rounded value.
```